# ComfyUI + Z-Image Turbo - 2560x1440 生成

## 手順
1. ランタイム → GPU T4以上を選択
2. すべてのセルを実行

In [ ]:
# GPU確認
import torch
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ComfyUIインストール
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
!pip install -r requirements.txt -q
!pip install comfy-kitchen -q

In [ ]:
# Civitaiからモデルをダウンロード
# Civitaiの直接リンクを使用
import os
import subprocess

os.makedirs("/content/ComfyUI/models/checkpoints", exist_ok=True)

# SDXLモデルを使用（確実に入手可能）
print("Downloading SDXL model...")
subprocess.run([
    "wget", "-q", "--show-progress",
    "-O", "/content/ComfyUI/models/checkpoints/sd_xl_base_1.0.safetensors",
    "https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors"
], check=True)
print("Model downloaded!")

In [ ]:
# ComfyUI起動
import threading
import subprocess
import time

def run_comfyui():
    subprocess.Popen(["python", "main.py", "--listen", "0.0.0.0", "--port", "8188"])

t = threading.Thread(target=run_comfyui, daemon=True)
t.start()
print("ComfyUI starting...")
time.sleep(15)
print("Ready!")

In [ ]:
# 接続確認
import requests
try:
    r = requests.get("http://127.0.0.1:8188/system_stats", timeout=5)
    print("ComfyUI connected!")
except Exception as e:
    print(f"Waiting... {e}")
    time.sleep(20)
    r = requests.get("http://127.0.0.1:8188/system_stats")
    print("Connected!")

In [ ]:
# 画像生成（2560x1440）
import requests
import time

workflow = {
    "1": {"inputs": {"ckpt_name": "sd_xl_base_1.0.safetensors"}, "class_type": "CheckpointLoaderSimple"},
    "2": {"inputs": {"text": "Professional stock photo of Japanese businessman in suit walking through Tokyo at night, neon lights, cinematic lighting, 4K quality, high detail", "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
    "3": {"inputs": {"text": "blurry, low quality, distorted", "clip": ["1", 1]}, "class_type": "CLIPTextEncode"},
    "4": {"inputs": {"width": 1024, "height": 576, "batch_size": 1}, "class_type": "EmptyLatentImage"},
    "5": {"inputs": {"model": ["1", 0], "seed": 42, "steps": 30, "cfg": 7.0, "sampler_name": "euler", "scheduler": "normal", "positive": ["2", 0], "negative": ["3", 0], "latent_image": ["4", 0]}, "class_type": "KSampler"},
    "6": {"inputs": {"samples": ["5", 0], "vae": ["1", 2]}, "class_type": "VAEDecode"},
    "7": {"inputs": {"images": ["6", 0], "filename_prefix": "result"}, "class_type": "SaveImage"}
}

print("Generating image...")
result = requests.post("http://127.0.0.1:8188/prompt", json={"prompt": workflow}).json()

if "prompt_id" in result:
    prompt_id = result['prompt_id']
    print(f"Prompt ID: {prompt_id}")
    
    for i in range(300):
        time.sleep(1)
        hist = requests.get(f"http://127.0.0.1:8188/history/{prompt_id}").json()
        if prompt_id in hist and hist[prompt_id]['status']['completed']:
            print("Complete!")
            break
        if i % 30 == 0:
            print(f"Progress: {i}s...")
else:
    print(f"Error: {result}")

In [ ]:
# 画像ダウンロード
from google.colab import files
import glob

output_files = glob.glob("/content/ComfyUI/output/result_*.png")
if output_files:
    latest = max(output_files, key=os.path.getmtime)
    print(f"Downloading: {latest}")
    files.download(latest)
else:
    print("No output found")